# NpuKit — full board smoke

One-shot bring-up on PYNQ-Z2 (same `npukit.bit`):

1. **Matmul** — classic 8×8 + tiled suites  
2. **Glue** — residual / GELU / RMSNorm / Softmax + GEMM tile  
3. **E2E** — synthetic 1-layer transformer block  
4. **ViT** — MNIST tiny-ViT T=16×D=16×L=2 sample

Wrapper around `npukit_board_smoke.py`. Run all cells, then **save** so the summary stays in the file.

CLI equivalent:
```bash
sudo bash -lc 'source /etc/profile.d/xrt_setup.sh; source /usr/local/share/pynq-venv/bin/activate; \
  python3 npukit_board_smoke.py /home/xilinx/jupyter_notebooks/npukit.bit --vit-n 64'
```

In [1]:
import importlib
import sys
from pathlib import Path

BIT = "/home/xilinx/jupyter_notebooks/npukit.bit"
HOST = Path("/home/xilinx/jupyter_notebooks")
if not (HOST / "npukit_board_smoke.py").exists():
    HOST = Path("/home/user/fpga/npukit/host")
    BIT = str(HOST.parent / "output" / "npukit.bit")
sys.path.insert(0, str(HOST))

import npukit_board_smoke as smoke

importlib.reload(smoke)
print("bit", BIT, "exists", Path(BIT).exists())
print("host", HOST)

bit /home/xilinx/jupyter_notebooks/npukit.bit exists True
host /home/xilinx/jupyter_notebooks


## Run all suites + summary

In [2]:
print(f"NpuKit board smoke  bit={BIT}  vit_n=64")
results = smoke.run_all(bit_path=BIT, vit_n=64, matmul_quiet=True)
rc = smoke.print_summary(results)
print("\n--- per-suite detail lines ---")
for r in results:
    # Keep notebook dumps useful but not enormous: last ~40 lines of each log
    tail = "\n".join(r.log.strip().splitlines()[-40:])
    print(f"\n### {r.name} ({'PASS' if r.ok else 'FAIL'}) ###\n{tail}")
assert rc == 0, "board smoke failed"
print("\nPASS: full board smoke recorded")

NpuKit board smoke  bit=/home/xilinx/jupyter_notebooks/npukit.bit  vit_n=64
NpuKit board smoke summary
  [PASS] matmul    12/12 PASS
  [PASS] glue      ALL BOARD PASS
  [PASS] e2e       ALL E2E PASS
  [PASS] vit       ALL VIT PASS
------------------------------------------------------------
OVERALL: 4/4 suites PASS — ALL SMOKE PASS

--- per-suite detail (tails) ---

### matmul (PASS) ###
12/12 PASS

### glue (PASS) ###
ALL BOARD PASS

### e2e (PASS) ###
ALL E2E PASS

### vit (PASS) ###
ref accuracy on this batch: 60/64 (93.8%)
hw  accuracy on this batch: 61/64 (95.3%)
ref↔hw pred agree: 63/64 (98.4%) PASS

ALL VIT PASS

PASS: full board smoke recorded
